# ViFinQA — BGE-M3 Synthetic Retriever Diagnostic V1

Chẩn đoán candidate BGE-M3 khi issuer-held-out thất bại. Notebook đo đồng thời synthetic `train`, `validation`, `test` để phân biệt **train-fit nhưng không generalise** với **objective/optimization collapse**.

**Ràng buộc:** không đọc benchmark questions, không tạo submission, không promotion model. `train` chỉ được mở sau explicit diagnostic opt-in trong CLI.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

REPO_DIR = Path('/kaggle/working/AI_guru_retriever_diagnostic')
SOURCE_TREE_NAME = 'ai_guru_synthetic_retriever_source_v1'
SOURCE_MANIFEST_NAME = 'ai_guru_synthetic_retriever_source_v1.manifest.json'
source_dirs = sorted({path.parent for path in Path('/kaggle/input').rglob(SOURCE_MANIFEST_NAME) if (path.parent / SOURCE_TREE_NAME).is_dir()})
if len(source_dirs) != 1:
    raise RuntimeError(f'Expected exactly one verified source snapshot; found {len(source_dirs)}.')
source_dir = source_dirs[0]
source_manifest = json.loads((source_dir / SOURCE_MANIFEST_NAME).read_text(encoding='utf-8'))
source_root = source_dir / SOURCE_TREE_NAME
platform_ignored_paths = {'pax_global_header'}
actual_files = {path.relative_to(source_root).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest() for path in source_root.rglob('*') if path.is_file() and path.relative_to(source_root).as_posix() not in platform_ignored_paths}
actual_tree_sha = hashlib.sha256(json.dumps(actual_files, sort_keys=True, separators=(',', ':')).encode()).hexdigest()
if source_manifest.get('protocol') != 'kaggle_synthetic_retriever_source_v1' or actual_tree_sha != source_manifest.get('source_tree_sha256') or len(actual_files) != source_manifest.get('source_tree_file_count'):
    raise ValueError('Synthetic source tree failed manifest/hash verification.')
if any(path == 'data/ViFinQA' or path.startswith('data/ViFinQA/') for path in actual_files):
    raise ValueError('Synthetic source snapshot must not contain benchmark data.')
if REPO_DIR.exists():
    raise RuntimeError(f'Refusing to merge source snapshot into existing path: {REPO_DIR}')
shutil.copytree(source_root, REPO_DIR)
diagnostic = REPO_DIR / 'scripts' / 'diagnose_synthetic_retriever_v1.py'
if not diagnostic.is_file():
    raise FileNotFoundError('Verified source snapshot is missing the retriever diagnostic script.')
print({'source_commit': source_manifest.get('git_commit'), 'source_tree_files': len(actual_files), 'diagnostic': str(diagnostic)})


In [ ]:
# Pin the same CUDA-compatible stack used for fine-tuning before importing model code.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'torch==2.12.1', 'torchvision==0.27.1', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'sentence-transformers==3.4.1', 'transformers==4.48.3'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR), '--no-deps'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator, restart, then Run All.')
vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gib < 14:
    raise RuntimeError(f'Retriever diagnostic requires >=14 GiB VRAM; found {vram_gib:.1f} GiB.')
print({'gpu': torch.cuda.get_device_name(0), 'vram_gib': round(vram_gib, 2), 'torch': torch.__version__})


## Inputs and diagnostic plan

Attach the synthetic curriculum Dataset, independent baseline `table_assets.jsonl`, verified source snapshot, and the completed training-kernel output. The script verifies every synthetic row, hashes its inputs, and evaluates base/fine-tuned models on the same full table corpus.

Read the result as follows: high train Recall and good positive-vs-hard-negative margin but zero held-out Recall indicates generalisation failure; poor train margin/Recall too indicates objective or optimisation collapse. Both outcomes remain non-promotable.

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
CURRICULUM_NAME = 'synthetic_finance_curriculum_v1.jsonl'
CURRICULUM_MANIFEST_NAME = 'synthetic_finance_curriculum_v1.manifest.json'
curriculum_dirs = sorted({path.parent for path in INPUT_ROOT.rglob(CURRICULUM_NAME) if (path.parent / CURRICULUM_MANIFEST_NAME).is_file()})
if len(curriculum_dirs) != 1:
    raise RuntimeError(f'Expected exactly one curriculum input directory; found {len(curriculum_dirs)}.')
CURRICULUM_DIR = curriculum_dirs[0]
CURRICULUM = CURRICULUM_DIR / CURRICULUM_NAME
CURRICULUM_MANIFEST = CURRICULUM_DIR / CURRICULUM_MANIFEST_NAME
table_assets = sorted(INPUT_ROOT.rglob('table_assets.jsonl'))
if len(table_assets) != 1:
    raise RuntimeError(f'Expected exactly one independent table_assets.jsonl input; found {len(table_assets)}.')
TABLES = table_assets[0]
model_dirs = sorted({path.parent for path in INPUT_ROOT.rglob('training_metadata.json') if (path.parent / 'model.safetensors').is_file() and (path.parent / 'modules.json').is_file()})
if len(model_dirs) != 1:
    raise RuntimeError(f'Expected exactly one trained model output directory; found {len(model_dirs)}.')
FINETUNED_MODEL = model_dirs[0]
training_metadata = json.loads((FINETUNED_MODEL / 'training_metadata.json').read_text(encoding='utf-8'))
if training_metadata.get('provenance') != 'synthetic_execution_verified':
    raise ValueError('Trained model metadata has an unexpected provenance.')
print({'curriculum': str(CURRICULUM), 'tables': str(TABLES), 'finetuned_model': str(FINETUNED_MODEL), 'training_examples': training_metadata.get('training_examples')})


In [ ]:
OUTPUT_DIR = Path('/kaggle/working/bge_m3_synthetic_v1_diagnostic')
command = [
    sys.executable, str(diagnostic),
    '--curriculum', str(CURRICULUM),
    '--manifest', str(CURRICULUM_MANIFEST),
    '--bundle-tables', str(TABLES),
    '--output-dir', str(OUTPUT_DIR),
    '--model', 'base=BAAI/bge-m3',
    '--model', f'finetuned={FINETUNED_MODEL}',
    '--splits', 'train', 'validation', 'test',
    '--diagnostic-allow-train',
    '--ks', '1', '3', '5', '10', '20',
    '--passage-batch-size', '16', '--query-batch-size', '32',
    '--max-seq-length', '384', '--device', 'cuda:0',
]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=REPO_DIR, check=True)


In [ ]:
result = json.loads((OUTPUT_DIR / 'diagnostic_manifest.json').read_text(encoding='utf-8'))
if result.get('diagnostic_status') != 'complete_not_for_promotion_or_submission':
    raise ValueError('Unexpected diagnostic status.')
summary = {
    label: {
        split: {
            'mrr': round(metrics['retrieval']['mrr'], 4),
            'recall@1': round(metrics['retrieval']['recall_at_k']['1'], 4),
            'recall@10': round(metrics['retrieval']['recall_at_k']['10'], 4),
            'pairwise_margin': round(metrics['pairwise']['positive_minus_hard_negative']['mean'], 4),
            'positive_beats_hard_negative_rate': round(metrics['pairwise']['positive_beats_hard_negative_rate'], 4),
        }
        for split, metrics in model_result['splits'].items()
    }
    for label, model_result in result['models'].items()
}
print(json.dumps({'summary': summary, 'status': result['diagnostic_status']}, ensure_ascii=False, indent=2))
print({'artifact_files': sorted(path.name for path in OUTPUT_DIR.iterdir())})


## Decision boundary

This diagnostic answers why the candidate failed; it does not make a candidate valid. Retain `offline_evaluation_complete_not_promoted` from the independent issuer-held-out evaluation. Build a new, separately named candidate only after reviewing this artifact; evaluate the replacement with the original held-out notebook.